In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [2]:
#_____TASK 1______

In [3]:
np.random.seed(42)

def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95 % CI).
    """

    data = np.asarray(data)
    n = len(data)
    boot_stats = np.zeros(n_boot)
    for i in range(n_boot):
        sample = np.random.choice(data, size=n, replace=True)
        boot_stats[i] = stat_func(sample)

    alpha = 1 - ci_level

    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))

    return lower, upper

data = np.arange(1, 101)

ci = bootstrap_ci(data)

print(ci)

(np.float64(44.799749999999996), np.float64(56.100249999999996))


In [4]:
#_____TASK 2_____

In [5]:
np.random.seed(42)

n = 200

income = np.random.normal(loc=5000, scale=1500, size=n)

purchased = np.random.binomial(n=1, p=0.4, size=n)

df = pd.DataFrame({
    'Income': income,
    'Purchased': purchased
})

print(df.head())

mean = np.mean(df["Income"])
median = np.median(df["Income"])
proportion = np.mean(df["Purchased"])

cl_df = pd.DataFrame({
    'Mean' : [mean],
    'Median' : [median],
    'Proportion' : [proportion]
})

print("")

print(cl_df)

        Income  Purchased
0  5745.071230          1
1  4792.603548          1
2  5971.532807          1
3  7284.544785          1
4  4648.769938          0

          Mean       Median  Proportion
0  4938.843552  4993.712174        0.42


In [6]:
#_____TASK 3_____

In [7]:
mean_boot = bootstrap_ci(df["Income"])
prop_boot = bootstrap_ci(df["Purchased"])

mean_val = df["Income"].mean()
se_mean = df["Income"].std(ddof=1) / np.sqrt(n)

mean_norm = st.t.interval(
    0.95,
    df=n-1,
    loc=mean_val,
    scale=se_mean
)

p_hat = df["Purchased"].mean()
se_p = np.sqrt(p_hat * (1 - p_hat) / n)

z = 1.96

prop_norm = (
    p_hat - z * se_p,
    p_hat + z * se_p
)

comparison = pd.DataFrame({
    "Statistic" : ["Mean", "Proportion"],
    "Bootstrap Lower" : [mean_boot[0], prop_boot[0]],
    "Bootstrap Upper" : [mean_boot[-1], prop_boot[-1]],
    "Normal Lower" : [mean_norm[0], prop_norm[0]],
    "Normal Upper" : [mean_norm[-1], prop_norm[-1]]
})

print(comparison)

print("")

"""
1. two approaches give almost similar intervals for the mean and proportion. They diverge more for statistics like the median, where the normal-approximation formula does not work well

2. bootstrap approach is especially useful for the median, because it does not assume a normal distribution and can estimate the interval for any statistic
"""

    Statistic  Bootstrap Lower  Bootstrap Upper  Normal Lower  Normal Upper
0        Mean      4744.894784      5131.303233   4744.117029   5133.570075
1  Proportion         0.355000         0.490000      0.351596      0.488404



'\n1. two approaches give almost similar intervals for the mean and proportion. They diverge more for statistics like the median, where the normal-approximation formula does not work well.\n\n2. bootstrap approach is especially useful for the median, because it does not assume a normal distribution and can estimate the interval for any statistic.\n'